# ONG v3 — Test DINOv2 ViT-L/14 (Gradio)

Dua-tahap, sama seperti app produksi tapi backbone-nya **DINOv2 ViT-L/14 @448** (pilot `best_model_global.pth`):

1. **Stage 1 — Genus** : classifier head → top-5 genus (softmax).
2. **Stage 2 — Spesies** : pre-logits embedding (1024-dim) + FAISS cosine → top-5 spesies × 5 foto termirip.

**Angka eval (frozen test split):** global top-1 **88.9%**, macro top-1 66.9% (58 genus di test), **genus R@5 98.7%**, species R@5 86.6%. Kekuatan model ini ada di **retrieval** — jadi Stage 2 yang paling andal.

**Yang harus ada di Drive `MyDrive/ONG_v3/`:** `models/dinov2l/best_model_global.pth`, `models/dinov2l/vocab.json`, `data/splits/all_images.csv`, `photos.zip`.

Pakai GPU (L4) — **Runtime → Change runtime type → L4**, lalu **Run all**. Build index spesies pertama kali ~3–5 menit, lalu di-cache ke Drive (run berikutnya instan).

In [ ]:
# Sel 1 — Install deps + cek GPU
!pip -q install -U timm faiss-cpu gradio
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE (pilih runtime L4)')

In [ ]:
# Sel 2 — Mount Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Sel 3 — Konfigurasi & cek file
import os
from pathlib import Path

ROOT       = '/content/drive/MyDrive/ONG_v3'
MODEL      = 'dinov2l'
EVAL_NAME  = 'vit_large_patch14_reg4_dinov2.lvd142m'   # timm model name (harus = saat training)
IMG        = 448
CKPT_TAG   = 'global'                                  # 'global' (top-1 terbaik) atau 'macro'

MODEL_DIR  = Path(f'{ROOT}/models/{MODEL}')
CKPT       = MODEL_DIR / f'best_model_{CKPT_TAG}.pth'
VOCAB      = MODEL_DIR / 'vocab.json'
ALL_CSV    = Path(f'{ROOT}/data/splits/all_images.csv')
PHOTOS_DIR = '/content/photos'
# Cache index spesies (per-model, per-tag) supaya run berikutnya tak rebuild
CACHE_DIR  = Path(f'{ROOT}/models/{MODEL}/retrieval_{CKPT_TAG}')

need = [CKPT, VOCAB, ALL_CSV, Path(f'{ROOT}/photos.zip')]
for f in need:
    print(f"[{'OK' if f.exists() else 'MISSING'}] {f}")
missing = [str(f) for f in need if not f.exists()]
assert not missing, f'Upload dulu ke Drive: {missing}'

In [ ]:
# Sel 4 — Unzip photos.zip → /content/photos/{Genus}/*.jpg
if os.path.isdir(PHOTOS_DIR) and len(os.listdir(PHOTOS_DIR)) >= 100:
    print(f'Sudah ter-extract: {len(os.listdir(PHOTOS_DIR))} folder genus — skip unzip.')
else:
    !unzip -q -o "{ROOT}/photos.zip" -d /content/
    # Perbaiki kalau ter-nested jadi /content/photos/photos
    nested = '/content/photos/photos'
    if os.path.isdir(nested):
        import shutil
        for it in os.listdir(nested):
            shutil.move(os.path.join(nested, it), PHOTOS_DIR)
        os.rmdir(nested)
    assert os.path.isdir(PHOTOS_DIR), 'Tidak ada /content/photos setelah unzip — cek layout zip.'
    print('Folder genus:', len(os.listdir(PHOTOS_DIR)))

In [ ]:
# Sel 5 — Load DINOv2 (preprocessing PERSIS seperti 13_evaluate.py)
import json, numpy as np
import torch, timm
import torch.nn.functional as F
from torchvision import transforms
from PIL import Image, ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
genera = json.loads(VOCAB.read_text())
n_cls  = len(genera)

# ViT: img_size HARUS diteruskan agar grid pos-embed cocok dgn checkpoint @448
model = timm.create_model(EVAL_NAME, pretrained=False, num_classes=n_cls, img_size=IMG)
cfg   = timm.data.resolve_model_data_config(model)
MEAN, STD = list(cfg['mean']), list(cfg['std'])

state = torch.load(CKPT, map_location='cpu')
state = state.get('state_dict', state) if isinstance(state, dict) else state
missing, unexpected = model.load_state_dict(state, strict=False)
print(f'load_state_dict: missing={len(missing)} unexpected={len(unexpected)}')
model = model.to(DEVICE).eval()

tfm = transforms.Compose([
    transforms.Resize(int(IMG * 1.1)),
    transforms.CenterCrop(IMG),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
USE_AMP = DEVICE.type == 'cuda'

@torch.no_grad()
def embed_logits(tensor, want_logits=True):
    """Pre-logits embedding (1024-d) + optional logits — mirror _emb_and_logits()."""
    with torch.autocast(DEVICE.type, enabled=USE_AMP):
        feat   = model.forward_features(tensor)
        emb    = model.forward_head(feat, pre_logits=True)
        logits = model.get_classifier()(emb) if want_logits else None
    return emb.float(), (logits.float() if logits is not None else None)

print(f'Model: {EVAL_NAME} | {n_cls} genus | img {IMG} | mean/std {MEAN}/{STD} | {DEVICE}')

In [ ]:
# Sel 6 — Build (atau load) reference embeddings + FAISS index spesies
import faiss, pandas as pd
from torch.utils.data import Dataset, DataLoader

EMB_PATH   = CACHE_DIR / 'ref_emb.npy'
META_PATH  = CACHE_DIR / 'metadata.json'
INDEX_PATH = CACHE_DIR / 'species_index.faiss'

def remap(p):
    parts = str(p).replace('\\', '/').split('/'); low = [x.lower() for x in parts]
    if 'photos' in low:
        i = low.index('photos'); return PHOTOS_DIR + '/' + '/'.join(parts[i + 1:])
    return p

class ImgDS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try:
            return tfm(Image.open(self.paths[i]).convert('RGB')), i
        except Exception:
            return torch.zeros(3, IMG, IMG), i

if EMB_PATH.exists() and META_PATH.exists():
    ref_emb  = np.load(EMB_PATH)
    metadata = json.loads(META_PATH.read_text(encoding='utf-8'))
    print(f'Loaded cache: {ref_emb.shape[0]:,} vektor (dim {ref_emb.shape[1]})')
else:
    df = pd.read_csv(ALL_CSV)
    df['path'] = df['path'].apply(remap)
    df = df[df['path'].apply(os.path.exists)].reset_index(drop=True)
    print(f'Embedding {len(df):,} foto referensi (±3–5 menit @L4)...')
    loader = DataLoader(ImgDS(df['path'].tolist()), batch_size=32, shuffle=False,
                        num_workers=2, pin_memory=True)
    chunks, order = [], []
    from tqdm.auto import tqdm
    for imgs, idx in tqdm(loader):
        emb, _ = embed_logits(imgs.to(DEVICE, non_blocking=True), want_logits=False)
        chunks.append(emb.cpu().numpy()); order.append(idx.numpy())
    order   = np.concatenate(order)
    ref_emb = np.concatenate(chunks)[np.argsort(order)].astype('float32')
    faiss.normalize_L2(ref_emb)                       # cosine via inner-product
    metadata = df[['species', 'genus', 'path']].to_dict('records')
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    np.save(EMB_PATH, ref_emb)
    META_PATH.write_text(json.dumps(metadata, ensure_ascii=False), encoding='utf-8')
    print(f'Saved cache → {CACHE_DIR}')

# --- FAISS ---
DIM = ref_emb.shape[1]
# Index global (cosine = inner-product krn vektor sudah L2-norm), disimpan ke Drive utk reuse
if INDEX_PATH.exists():
    faiss_index = faiss.read_index(str(INDEX_PATH))
else:
    faiss_index = faiss.IndexFlatIP(DIM)
    faiss_index.add(ref_emb)
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    faiss.write_index(faiss_index, str(INDEX_PATH))

from collections import defaultdict
genus_to_idx = defaultdict(list)
for i, rec in enumerate(metadata):
    genus_to_idx[rec['genus']].append(i)

# Sub-index FAISS per genus (dibangun sekali) → Stage 2 mencari di dalam genus prediksi
genus_index = {}
for g, idxs in genus_to_idx.items():
    sub = faiss.IndexFlatIP(DIM)
    sub.add(np.ascontiguousarray(ref_emb[idxs]))
    genus_index[g] = (sub, np.asarray(idxs))

print(f'FAISS siap: global {faiss_index.ntotal:,} vektor (dim {DIM}), '
      f'{len(genus_index)} sub-index per-genus.')

In [ ]:
# Sel 7 — Fungsi inferensi (dua-tahap, retrieval pakai FAISS)
PLACEHOLDER = Image.new('RGB', (300, 300), (220, 230, 220))

def open_photo(path):
    try:
        return Image.open(path).convert('RGB')
    except Exception:
        return PLACEHOLDER

@torch.no_grad()
def identify(img, top_n=5, photos_per_species=5, n_genera=3):
    tensor = tfm(img.convert('RGB')).unsqueeze(0).to(DEVICE)
    emb, logits = embed_logits(tensor, want_logits=True)
    probs = F.softmax(logits, dim=1)[0]
    top5_p, top5_i = probs.topk(5)
    q = emb.cpu().numpy().astype('float32'); faiss.normalize_L2(q)   # query utk FAISS

    # Stage 2: cari di sub-index FAISS dari top-N genus prediksi, kumpulkan per spesies
    pool = defaultdict(list)
    for g_prob, g_idx in zip(top5_p[:n_genera].tolist(), top5_i[:n_genera].tolist()):
        genus = genera[g_idx]
        if genus not in genus_index:
            continue
        sub, idxs = genus_index[genus]
        k = min(200, sub.ntotal)                       # ambil banyak utk cover >1 spesies/genus
        D, I = sub.search(q, k)                         # FAISS inner-product = cosine
        for score, local in zip(D[0], I[0]):
            if local < 0:
                continue
            rec = metadata[int(idxs[local])]
            pool[rec['species']].append({'genus': genus, 'genus_conf': g_prob,
                                         'score': g_prob * float(score),
                                         'sim': float(score), 'path': rec['path']})
    for sp in pool:
        pool[sp].sort(key=lambda x: x['score'], reverse=True)
        pool[sp] = pool[sp][:photos_per_species]
    ranked = sorted(pool.items(), key=lambda kv: kv[1][0]['score'], reverse=True)[:top_n]

    results = [{'species': sp, 'genus': ph[0]['genus'], 'genus_conf': ph[0]['genus_conf'],
                'best_score': ph[0]['score'], 'photos': ph} for sp, ph in ranked]
    genus_labels = {genera[i.item()]: float(p.item()) for p, i in zip(top5_p, top5_i)}
    return results, genus_labels

def predict(img):
    if img is None:
        return {}, [], 'Upload foto anggrek dulu.'
    results, genus_labels = identify(img)
    gallery = []
    for rank, res in enumerate(results, 1):
        for j, ph in enumerate(res['photos'], 1):
            cap = (f"#{rank} {res['species']}  [{j}/{len(res['photos'])}]\n"
                   f"Genus: {res['genus']} ({res['genus_conf']*100:.0f}%) | sim {ph['sim']:.3f}")
            gallery.append((open_photo(ph['path']), cap))
    best = results[0] if results else None
    summary = (f"### {best['species']}\nGenus **{best['genus']}** · "
               f"conf {best['genus_conf']*100:.0f}% · sim {best['photos'][0]['sim']:.3f}"
               if best else 'Tidak ada hasil.')
    return genus_labels, gallery, summary

print('Fungsi siap.')

In [ ]:
# Sel 8 — Launch Gradio (share link publik ~72 jam)
import gradio as gr

with gr.Blocks(title='ONG v3 — DINOv2 Test') as demo:
    gr.Markdown(
        '# ONG v3 — DINOv2 ViT-L/14 (test)\n'
        '**120 genus** · global top-1 88.9% · **genus R@5 98.7%** · species R@5 86.6%\n\n'
        'Stage 1 prediksi genus · Stage 2 retrieval top-5 spesies × 5 foto termirip.'
    )
    with gr.Row():
        with gr.Column(scale=1):
            img_input   = gr.Image(type='pil', label='Upload Foto Anggrek')
            run_btn     = gr.Button('Identify', variant='primary')
            summary_out = gr.Markdown()
        with gr.Column(scale=2):
            genus_out = gr.Label(num_top_classes=5, label='Stage 1 — Top-5 Genus')
    gallery_out = gr.Gallery(label='Stage 2 — Top-5 Spesies × 5 Foto',
                             columns=5, rows=5, height=650, object_fit='cover')
    run_btn.click(predict, img_input, [genus_out, gallery_out, summary_out])
    img_input.change(predict, img_input, [genus_out, gallery_out, summary_out])

demo.launch(share=True, debug=False)